# Reading a Fitted Tree

**Notebook 4 of 5.** A decision tree is the one model you can read end to end. This notebook
reads one twice:

- **a single recipe**, followed question by question down to the leaf that predicts it;
- **the whole model**, asking which ingredients it relies on overall.

The second reading has a trap in it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.inspection import permutation_importance

## 1. The data, and the tree we will read

Notebook 03's concrete: **1,030 laboratory mixes**, seven ingredients measured in kilograms
per cubic meter, the sample's age in days, and `compressive_strength` in MPa as the target.

The split and the model are notebook 03's too, so nothing here is new except the reading.

In [ ]:
concrete = pd.read_csv("../data/concrete.csv")
X = concrete.drop(columns="compressive_strength")
y = concrete["compressive_strength"]

# the same split and the same kind of tree as notebook 03; depth 9 sits on the flat
# part of its validation curve, so anything from 8 upward would do
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
tree = DecisionTreeRegressor(max_depth=9, random_state=42)
tree.fit(X_train, y_train)

## 2. One recipe, all the way down

Take the first row of the test set — one recipe for concrete, which this notebook calls a
**mix**.

In [ ]:
# the mix the tree is about to be asked about, one column so it reads easily
mix = X_test.iloc[[0]]
mix

Those eight numbers are the whole input. Every question below is about one of them, so the
diagram and this table can be read side by side.

`decision_path` returns the nodes the mix travels through, and every node but the last holds
a question.

In [ ]:
inner = tree.tree_

# walk the path, collecting one question per node until the leaf ends it
questions = []
for node in tree.decision_path(mix).indices:

    # children_left is -1 for a leaf, which is where the path stops
    if inner.children_left[node] == -1:
        prediction = inner.value[node][0][0]
        leaf_rows = int(inner.weighted_n_node_samples[node])
        break

    feature = X.columns[inner.feature[node]]
    threshold = inner.threshold[node]
    value = mix.iloc[0][feature]
    questions.append((feature, threshold, value, value <= threshold))

DARK, ORANGE = "#252629", "#FF4A11"
fig, ax = plt.subplots(figsize=(7.5, 0.95 * (len(questions) + 1)))

for position, (feature, threshold, value, went_left) in enumerate(questions):
    height = -position
    ax.text(0.0, height, rf"{feature} $\leq$ {threshold:.1f} ?", ha="center", va="center",
            fontsize=11, bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor=DARK))

    # the arrow to the next question, labelled with this mix's answer
    ax.annotate("", xy=(0.0, height - 0.72), xytext=(0.0, height - 0.28),
                arrowprops=dict(arrowstyle="-|>", color=DARK, lw=1.2))
    ax.text(0.06, height - 0.5, f"{value:.1f}, so {'yes' if went_left else 'no'}",
            ha="left", va="center", fontsize=9.5, color=DARK)

ax.text(0.0, -len(questions), f"predict {prediction:.2f} MPa\nmean of {leaf_rows} training mixes",
        ha="center", va="center", fontsize=11, color="white",
        bbox=dict(boxstyle="round,pad=0.5", facecolor=ORANGE, edgecolor=ORANGE))

ax.set_xlim(-0.5, 0.5); ax.set_ylim(-len(questions) - 0.6, 0.6)
ax.axis("off")
ax.set_title(f"one mix, measured at {y_test.iloc[0]:.2f} MPa", fontsize=11, color=DARK)
plt.show()

**Nine questions, then an answer.** Nothing is hidden: this mix is predicted at 50.18 MPa
because twelve training mixes answered the same nine questions the same way, and their
strengths averaged 50.18. It really measured 52.91.

Look at the left-hand column of the diagram. `cement` is asked about three times, `age`
twice, `fine_aggregate` twice, and three of the eight ingredients never come up at all.
That is one path. The whole tree is thousands of them.

## 3. From one path to every path

**`feature_importances_` is that counting, done over every node.** Each split lowers
impurity; credit it to the feature that asked, weight it by the rows passing through, and
scale the totals to 1. It is called **mean decrease in impurity**.

In [ ]:
importances = pd.Series(tree.feature_importances_, index=X.columns).sort_values()

importances.plot.barh(figsize=(7, 4), color="#FF4A11")
plt.xlabel("mean decrease in impurity")
plt.title("Which ingredients the tree split on")
plt.show()

importances.sort_values(ascending=False).round(4)

**Cement 0.364, age 0.323, water 0.141** — 83% between them, and chemistry a civil engineer
would recognize. Which is the trouble: it looks right, so it gets trusted.

## 4. A column of random numbers

> **Exercise:**
>
> Add two useless columns to train and test: `noise` from `rng.random(...)` and `coin` from
> `rng.integers(0, 2, ...)`. Refit the same tree and see where they land in the ranking.

<details><summary>
Click here for a hint...
</summary>

`rng = np.random.default_rng(0)`, copy `X_train` and `X_test`, assign both columns, refit
`DecisionTreeRegressor(max_depth=9, random_state=42)`.

</details>

In [ ]:
# a seeded generator, so the useless columns are the same on every run
rng = np.random.default_rng(0)

X_train_noisy, X_test_noisy = X_train.copy(), X_test.copy()

# 824 distinct values, so it offers the tree hundreds of thresholds to try
X_train_noisy["noise"] = rng.random(len(X_train_noisy))
X_test_noisy["noise"] = rng.random(len(X_test_noisy))

# two values, so it offers exactly one threshold
X_train_noisy["coin"] = rng.integers(0, 2, len(X_train_noisy))
X_test_noisy["coin"] = rng.integers(0, 2, len(X_test_noisy))

noisy_tree = DecisionTreeRegressor(max_depth=9, random_state=42)
noisy_tree.fit(X_train_noisy, y_train)

pd.Series(
    noisy_tree.feature_importances_, index=X_train_noisy.columns
).sort_values(ascending=False).round(4)

<details><summary>
Click here to compare your answer...
</summary>

**`noise` scores 0.0124, above `coarse_aggregate` at 0.0116.** Random numbers, ranked above a
real ingredient. `coin` scores 0.0010.

</details>

Every question on that path in section 2 was chosen by trying **every threshold on every
feature** and keeping the best, which is where the bias comes from:

- `noise` has 824 distinct values, so the root alone offers 823 thresholds. Any sample
  contains accidental patterns, and with that many cuts one of them always lowers training
  impurity a little — credited to `noise`.
- `coin` offers one threshold, so it almost never wins.

> **Cardinality bias.** Mean decrease in impurity favors features with many distinct values
> — continuous columns, identifiers, dates — whether or not they help.

It is why a customer ID can look like your best predictor.

## 5. Asking the other question

Impurity importance asks *how much work did this feature do while the tree was built*. For a
finished model there is a better question: **how much worse does it get if this column is
scrambled?**

> **Exercise:**
>
> Run `permutation_importance` on `noisy_tree` with the noisy test set, `n_repeats=10`,
> `random_state=42`, `scoring="r2"`, and rank it. Where do `noise` and `coin` land now?

<details><summary>
Click here for a hint...
</summary>

The returned object's `.importances_mean` is the average drop per column.

</details>

In [ ]:
# scored on the test set: the fitted tree never changes, only the data it sees
result = permutation_importance(
    noisy_tree, X_test_noisy, y_test, n_repeats=10, random_state=42, scoring="r2"
)

pd.Series(
    result.importances_mean, index=X_train_noisy.columns
).sort_values(ascending=False).round(4)

<details><summary>
Click here to compare your answer...
</summary>

**`noise` -0.0005, `coin` -0.0001** — zero, give or take the randomness of shuffling.

The real features also swap: **age (0.858) now outranks cement (0.754)**. The two measures
answer different questions, and where they disagree this is the one about predictions.

</details>

## 6. What an importance number loses

Both measures squeeze a model into one number per column:

- **correlated columns hide each other** — scramble one, the model recovers it from the other,
  and permutation importance calls both unimportant;
- **one-hot pieces look small** — notebook 02 split `occupation` into 14 columns, each
  holding a share. Add them back before reading them;
- **importance is not causation** — `age` leading does not mean curing a mix longer would
  strengthen it.

## Optional: the same number by hand

`tree_` holds the arrays the fitted tree is made of, so section 3's definition runs directly.
Each internal node contributes

$$n_{\text{node}} \, I_{\text{node}} - n_{\text{left}} \, I_{\text{left}} - n_{\text{right}} \, I_{\text{right}}$$

to the feature it split on, where $I$ is the node's impurity and $n$ the training rows that
reached it.

In [ ]:
totals = np.zeros(X.shape[1])

# every node in the tree, leaves included, in no particular order
for node in range(inner.node_count):
    left, right = inner.children_left[node], inner.children_right[node]
    if left == -1:
        continue
    n, n_left, n_right = inner.weighted_n_node_samples[[node, left, right]]
    totals[inner.feature[node]] += (
        n * inner.impurity[node] - n_left * inner.impurity[left] - n_right * inner.impurity[right]
    )

# dividing by the sum is what makes the importances add to 1
totals /= totals.sum()

pd.DataFrame({"by hand": totals, "scikit-learn": tree.feature_importances_}, index=X.columns).round(6)

Identical to six decimals — and note what the loop reads: `impurity` and
`weighted_n_node_samples`, both recorded while the tree was **trained**. That is why the
measure says what the tree did, not whether it generalized.

## Summary

- A fitted tree can be **read directly**: `decision_path` gives the questions one row
  answered, and its leaf gives the mixes the prediction was averaged from.
- **`feature_importances_`** is that counting over every node — computed on training data,
  and biased towards columns with many distinct values.
- **Permutation importance** scrambles a column in held-out data and measures the drop. It
  put both random columns at zero and reordered the real ones.
- Every importance number is a summary, and summaries lose things.

Next, in `05_exercise_full_cycle.ipynb`: the whole workflow, start to finish, on your own.